# Robustness Analysis

## 1. Main result and grading declaration

The primary econometric analysis examined the relationship between international student enrolments in Victoria and the estimated Melbourne rental vacancy rate using quarterly data from 2020 to 2025.

The main result from the primary analysis was that the estimated relationship between international student enrolments and the Melbourne rental vacancy rate was very small and statistically insignificant. This means that the available quarterly data do not provide strong statistical evidence of a stable positive or negative relationship.

This finding is interpreted as descriptive rather than causal. The analysis does not claim that changes in international student enrolments causally affect rental vacancy rates. Instead, it examines whether international student enrolments are conditionally associated with Melbourne rental vacancy rates in the available quarterly data.

The robustness analysis tests whether the main result is sensitive to alternative control sets, alternative samples, functional-form choices, and inference methods.

## 2. Load packages and clean data

This section imports the required packages, defines reproducible project paths, and loads the cleaned dataset generated by `code/clean_data.py`.

In [145]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from pathlib import Path

# Paths
ROOT = Path("..")
DATA_PATH = ROOT / "data" / "clean" / "clean_data.csv"
OUTPUT_DIR = ROOT / "outputs" / "tables"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load clean data
df = pd.read_csv(DATA_PATH)

print("Data shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

display(df.head())

Data shape: (24, 13)
Columns:
['year', 'quarter', 'quarter_end_month', 'quarter_end_date', 'vic_ytd_enrolments', 'vic_ytd_commencements', 'estimated_melbourne_vacancies', 'estimated_melbourne_vacancy_rate_pct', 'student_geography', 'vacancy_geography', 'notes', 'quarter_num', 'quarter_end_month_num']


,year,quarter,quarter_end_month,quarter_end_date,vic_ytd_enrolments,vic_ytd_commencements,estimated_melbourne_vacancies,estimated_melbourne_vacancy_rate_pct,student_geography,vacancy_geography,notes,quarter_num,quarter_end_month_num
0,2020,Q1,Mar,2020-03-31,216681,56991,3000,4.4,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,1,3
1,2020,Q2,Jun,2020-06-30,235433,75743,6200,8.0,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,2,6
2,2020,Q3,Sep,2020-09-30,267172,107482,8100,10.3,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,3,9
3,2020,Q4,Dec,2020-12-31,282720,123030,7800,9.9,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,4,12
4,2021,Q1,Mar,2021-03-31,178296,37476,6700,8.5,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,1,3


## 3. Inspect cleaned dataset

Before running robustness checks, this section inspects the structure of the cleaned dataset, including variable types, missing values, and summary statistics. This helps verify that the robustness analysis is based on the correct analysis-ready dataset.

In [146]:
print("Data types:")
display(df.dtypes)

print("Missing values:")
display(df.isna().sum().sort_values(ascending=False))

print("Summary statistics:")
display(df.describe(include="all"))

Data types:


year                                      int64
quarter                                     str
quarter_end_month                           str
quarter_end_date                            str
vic_ytd_enrolments                        int64
vic_ytd_commencements                     int64
estimated_melbourne_vacancies             int64
estimated_melbourne_vacancy_rate_pct    float64
student_geography                           str
vacancy_geography                           str
notes                                       str
quarter_num                               int64
quarter_end_month_num                     int64
dtype: object

Missing values:


year                                    0
quarter                                 0
quarter_end_month                       0
quarter_end_date                        0
vic_ytd_enrolments                      0
vic_ytd_commencements                   0
estimated_melbourne_vacancies           0
estimated_melbourne_vacancy_rate_pct    0
student_geography                       0
vacancy_geography                       0
notes                                   0
quarter_num                             0
quarter_end_month_num                   0
dtype: int64

Summary statistics:


,year,quarter,quarter_end_month,quarter_end_date,vic_ytd_enrolments,vic_ytd_commencements,estimated_melbourne_vacancies,estimated_melbourne_vacancy_rate_pct,student_geography,vacancy_geography,notes,quarter_num,quarter_end_month_num
count,24.000000,24,24,24,24.000000,24.000000,24.000000,24.000000,24,24,24,24.00000,24.000000
unique,NaN,4,4,24,NaN,NaN,NaN,NaN,1,1,1,NaN,NaN
top,NaN,Q1,Mar,2020-03-31,NaN,NaN,NaN,NaN,VIC,Melbourne,Quarter-end YTD student data; vacancy is chart...,NaN,NaN
freq,NaN,6,6,1,NaN,NaN,NaN,NaN,24,24,24,NaN,NaN
mean,2022.500000,NaN,NaN,NaN,237502.416667,93522.250000,2916.666667,3.791667,NaN,NaN,NaN,2.50000,7.500000
std,1.744557,NaN,NaN,NaN,49660.709950,39799.040087,2302.865260,2.901262,NaN,NaN,NaN,1.14208,3.426241
min,2020.000000,NaN,NaN,NaN,146792.000000,37476.000000,800.000000,1.100000,NaN,NaN,NaN,1.00000,3.000000
25%,2021.000000,NaN,NaN,NaN,204511.750000,59133.250000,1500.000000,2.000000,NaN,NaN,NaN,1.75000,5.250000
50%,2022.500000,NaN,NaN,NaN,231081.500000,84843.000000,1700.000000,2.250000,NaN,NaN,NaN,2.50000,7.500000
75%,2024.000000,NaN,NaN,NaN,271059.000000,123930.000000,3975.000000,5.075000,NaN,NaN,NaN,3.25000,9.750000


## 4. Define variables and construct analysis sample

The cleaned dataset contains quarterly observations from 2020 to 2025. The outcome variable is the estimated Melbourne rental vacancy rate. The main explanatory variable is Victorian year-to-date international student enrolments.

Because the dataset is a quarterly aggregate time series rather than an area-level panel, the robustness checks focus on alternative control sets, sample restrictions, functional-form choices, and inference methods. Area fixed effects and clustered standard errors by LGA are not appropriate for this dataset.

In [147]:
# Define key variables based on the cleaned dataset
OUTCOME = "estimated_melbourne_vacancy_rate_pct"
TREATMENT = "vic_ytd_enrolments"
COMMENCEMENTS = "vic_ytd_commencements"
QUARTER = "quarter"
YEAR = "year"

required_vars = [
    OUTCOME,
    TREATMENT,
    COMMENCEMENTS,
    YEAR,
    QUARTER,
    QUARTER_NUM,
    DATE
]

for var in required_vars:
    if var not in df.columns:
        raise ValueError(f"Missing required variable: {var}")
    else:
        print(f"Found variable: {var}")

# Convert date variable and sort chronologically
df[DATE] = pd.to_datetime(df[DATE], errors="raise")
df = df.sort_values([YEAR, QUARTER_NUM]).reset_index(drop=True)

# Create a linear time trend for robustness checks
df["time_trend"] = range(1, len(df) + 1)

# Construct regression sample
analysis_vars = [
    OUTCOME,
    TREATMENT,
    COMMENCEMENTS,
    YEAR,
    QUARTER,
    QUARTER_NUM,
    DATE,
    "time_trend"
]

reg_df = df[analysis_vars].dropna().copy()

print("Original observations:", len(df))
print("Regression sample observations:", len(reg_df))
print("Dropped observations due to missing values:", len(df) - len(reg_df))

display(reg_df.head())

Found variable: estimated_melbourne_vacancy_rate_pct
Found variable: vic_ytd_enrolments
Found variable: vic_ytd_commencements
Found variable: year
Found variable: quarter
Found variable: quarter_num
Found variable: quarter_end_date
Original observations: 24
Regression sample observations: 24
Dropped observations due to missing values: 0


,estimated_melbourne_vacancy_rate_pct,vic_ytd_enrolments,vic_ytd_commencements,year,quarter,quarter_num,quarter_end_date,time_trend
0,4.4,216681,56991,2020,Q1,1,2020-03-31,1
1,8.0,235433,75743,2020,Q2,2,2020-06-30,2
2,10.3,267172,107482,2020,Q3,3,2020-09-30,3
3,9.9,282720,123030,2020,Q4,4,2020-12-31,4
4,8.5,178296,37476,2021,Q1,1,2021-03-31,5


## 5. Analysis sample diagnostics

This section summarises the regression sample used for the robustness analysis. Since the sample contains only 24 quarterly observations, the results should be interpreted cautiously. The robustness checks are designed to test whether the main descriptive result is sensitive to seasonal controls, time trends, sample restrictions, outliers, functional-form choices, and alternative standard errors.

In [148]:
print("Number of quarterly observations:", len(reg_df))
print("Years covered:", reg_df[YEAR].min(), "to", reg_df[YEAR].max())
print("Quarters:", sorted(reg_df[QUARTER].unique()))

print("\nObservations by year:")
display(reg_df.groupby(YEAR).size())

print("\nSummary of key variables:")
display(reg_df[[OUTCOME, TREATMENT, COMMENCEMENTS, "time_trend"]].describe())

Number of quarterly observations: 24
Years covered: 2020 to 2025
Quarters: ['Q1', 'Q2', 'Q3', 'Q4']

Observations by year:


year
2020    4
2021    4
2022    4
2023    4
2024    4
2025    4
dtype: int64


Summary of key variables:


,estimated_melbourne_vacancy_rate_pct,vic_ytd_enrolments,vic_ytd_commencements,time_trend
count,24.000000,24.000000,24.000000,24.000000
mean,3.791667,237502.416667,93522.250000,12.500000
std,2.901262,49660.709950,39799.040087,7.071068
min,1.100000,146792.000000,37476.000000,1.000000
25%,2.000000,204511.750000,59133.250000,6.750000
50%,2.250000,231081.500000,84843.000000,12.500000
75%,5.075000,271059.000000,123930.000000,18.250000
max,10.300000,327815.000000,171667.000000,24.000000


## 6. Robustness check design

The robustness checks are selected to stress-test the main descriptive result from the primary econometric analysis.

The main specification uses international student enrolments as the explanatory variable and includes quarter fixed effects to account for seasonal differences across quarters.

The robustness checks are:

1. No controls: tests whether the result depends on including seasonal controls.
2. Add time trend: tests whether the relationship is driven by common time trends over 2020–2025.
3. Add commencements: tests whether the result changes when controlling for new international student commencements.
4. Restrict to 2022–2025: tests whether the result is driven by the unusual COVID-disrupted years of 2020–2021.
5. Drop outliers: tests whether the result is driven by extreme values of enrolments or vacancy rates.
6. Log enrolments: tests whether the result depends on using enrolments in levels rather than a nonlinear functional form.
7. HC3 standard errors: tests whether inference is sensitive to a more conservative heteroskedasticity-robust standard error estimator.

Because the analysis uses quarterly aggregate data rather than LGA-level panel data, clustered standard errors by area are not appropriate.

## 7. Regression helper functions

This section defines helper functions to estimate OLS models with robust standard errors and extract the coefficient of interest from each specification.

In [149]:
def fit_ols(formula, data, cov_type="HC1"):
    """
    Fit an OLS regression using the specified covariance estimator.
    """
    model = smf.ols(formula, data=data).fit(cov_type=cov_type)
    return model


def significance_stars(pval):
    """
    Return conventional significance stars based on p-value.
    """
    if pd.isna(pval):
        return ""
    elif pval < 0.01:
        return "***"
    elif pval < 0.05:
        return "**"
    elif pval < 0.10:
        return "*"
    else:
        return ""


def extract_model_result(model, variable):
    """
    Extract coefficient, standard error, p-value, and N for a target variable.
    """
    coef = model.params.get(variable, np.nan)
    se = model.bse.get(variable, np.nan)
    pval = model.pvalues.get(variable, np.nan)
    nobs = int(model.nobs)

    return {
        "coef": coef,
        "se": se,
        "pval": pval,
        "nobs": nobs,
        "stars": significance_stars(pval)
    }

## 8. Robustness specifications

This section estimates the main specification and a set of robustness checks. The coefficient of interest is the coefficient on international student enrolments, except in the log-enrolment specification where the coefficient of interest is the coefficient on log enrolments.

In [150]:
models = {}

# Column 1: Main specification from the primary econometric analysis
models["Main OLS"] = {
    "model": fit_ols(
        f"{OUTCOME} ~ {TREATMENT}",
        reg_df,
        cov_type="HC1"
    ),
    "variable": TREATMENT,
    "note": "Main primary-analysis specification with no additional controls."
}

# Column 2: Quarter fixed effects
models["Quarter FE"] = {
    "model": fit_ols(
        f"{OUTCOME} ~ {TREATMENT} + C({QUARTER})",
        reg_df,
        cov_type="HC1"
    ),
    "variable": TREATMENT,
    "note": "Adds quarter fixed effects to control for seasonal differences."
}

# Column 3: Add time trend
models["Add time trend"] = {
    "model": fit_ols(
        f"{OUTCOME} ~ {TREATMENT} + C({QUARTER}) + time_trend",
        reg_df,
        cov_type="HC1"
    ),
    "variable": TREATMENT,
    "note": "Adds quarter fixed effects and a linear time trend."
}

# Column 4: Add commencements control
models["Add commencements"] = {
    "model": fit_ols(
        f"{OUTCOME} ~ {TREATMENT} + {COMMENCEMENTS} + C({QUARTER})",
        reg_df,
        cov_type="HC1"
    ),
    "variable": TREATMENT,
    "note": "Adds quarter fixed effects and controls for international student commencements."
}

# Column 5: Restrict to 2022–2025
post_covid_df = reg_df[reg_df[YEAR] >= 2022].copy()

models["2022–2025 sample"] = {
    "model": fit_ols(
        f"{OUTCOME} ~ {TREATMENT} + C({QUARTER})",
        post_covid_df,
        cov_type="HC1"
    ),
    "variable": TREATMENT,
    "note": "Excludes 2020–2021 COVID-disrupted years."
}

# Column 6: Drop outliers using 5th–95th percentile rule
lower_enrol = reg_df[TREATMENT].quantile(0.05)
upper_enrol = reg_df[TREATMENT].quantile(0.95)
lower_vacancy = reg_df[OUTCOME].quantile(0.05)
upper_vacancy = reg_df[OUTCOME].quantile(0.95)

no_outlier_df = reg_df[
    reg_df[TREATMENT].between(lower_enrol, upper_enrol)
    & reg_df[OUTCOME].between(lower_vacancy, upper_vacancy)
].copy()

models["Drop outliers"] = {
    "model": fit_ols(
        f"{OUTCOME} ~ {TREATMENT} + C({QUARTER})",
        no_outlier_df,
        cov_type="HC1"
    ),
    "variable": TREATMENT,
    "note": "Drops observations outside the 5th–95th percentiles of enrolments and vacancy rate."
}

# Column 7: Functional form check using log enrolments
reg_df["log_enrolments"] = np.log1p(reg_df[TREATMENT])

models["Log enrolments"] = {
    "model": fit_ols(
        f"{OUTCOME} ~ log_enrolments + C({QUARTER})",
        reg_df,
        cov_type="HC1"
    ),
    "variable": "log_enrolments",
    "note": "Uses log(1 + enrolments) instead of enrolments in levels."
}

# Column 8: Alternative inference using HC3 standard errors for the main OLS model
models["HC3 standard errors"] = {
    "model": fit_ols(
        f"{OUTCOME} ~ {TREATMENT}",
        reg_df,
        cov_type="HC3"
    ),
    "variable": TREATMENT,
    "note": "Uses HC3 robust standard errors for the main OLS specification."
}

print("Robustness specifications estimated successfully.")

Robustness specifications estimated successfully.


## 9. Robustness table

The robustness table reports the coefficient of interest, robust standard error, p-value, and number of observations for each specification. The main specification appears first, followed by robustness checks.

In [151]:
rows = {}

for spec_name, item in models.items():
    model = item["model"]
    variable = item["variable"]
    result = extract_model_result(model, variable)

    rows[spec_name] = {
        "Coefficient": f"{result['coef']:.9f}{result['stars']}",
        "Std. Error": f"({result['se']:.9f})",
        "p-value": f"{result['pval']:.3f}",
        "N": result["nobs"],
        "Notes": item["note"]
    }

robustness_table = pd.DataFrame(rows)

display(robustness_table)

,Main OLS,Quarter FE,Add time trend,Add commencements,2022–2025 sample,Drop outliers,Log enrolments,HC3 standard errors
Coefficient,0.000000303,-0.000004852,0.000038930***,0.000055131**,0.000005485*,-0.000030277,-0.677478663,0.000000303
Std. Error,(0.000010772),(0.000012925),(0.000012431),(0.000024468),(0.000003079),(0.000019702),(2.976621581),(0.000011491)
p-value,0.978,0.707,0.002,0.024,0.075,0.124,0.820,0.979
N,24,24,24,24,16,17,24,24
Notes,Main primary-analysis specification with no ad...,Adds quarter fixed effects to control for seas...,Adds quarter fixed effects and a linear time t...,Adds quarter fixed effects and controls for in...,Excludes 2020–2021 COVID-disrupted years.,Drops observations outside the 5th–95th percen...,Uses log(1 + enrolments) instead of enrolments...,Uses HC3 robust standard errors for the main O...


## 10. Save outputs

This section saves the robustness table to the outputs folder so that the results can be reproduced and inspected outside the notebook.

In [152]:
robustness_table.to_csv(OUTPUT_DIR / "robustness_table.csv", index=True)
robustness_table.to_html(OUTPUT_DIR / "robustness_table.html", index=True)

print("Saved robustness table to:")
print(OUTPUT_DIR / "robustness_table.csv")
print(OUTPUT_DIR / "robustness_table.html")

Saved robustness table to:
..\outputs\tables\robustness_table.csv
..\outputs\tables\robustness_table.html


## 11. Interpretation

The robustness checks show that the primary result is sensitive to specification choices. In the main OLS specification from the primary econometric analysis, the coefficient on Victorian international student enrolments is very small and statistically insignificant. This means that the available quarterly data do not provide strong evidence of a meaningful contemporaneous relationship between international student enrolments and the estimated Melbourne rental vacancy rate.

Several robustness checks also produce statistically insignificant estimates, including the quarter fixed effects specification, the outlier-restricted sample, the log-enrolment specification, and the HC3 standard error check. These results are broadly consistent with the primary finding that the relationship is weak and imprecisely estimated.

However, the result changes in some important specifications. When a linear time trend is added, the coefficient becomes positive and statistically significant. The coefficient also becomes positive and statistically significant when international student commencements are added as an additional control, and it becomes positive and marginally significant in the 2022–2025 restricted sample. These changes suggest that the estimated relationship is sensitive to time trends, the treatment of COVID-disrupted years, and the choice of student-demand controls.

Overall, the robustness analysis does not provide robust evidence of a stable descriptive relationship between international student enrolments and Melbourne rental vacancy rates. The most defensible conclusion is that the current quarterly data are insufficient to support a strong positive or negative relationship, and the analysis should not be interpreted causally.